# Stage B4 — Export for iOS (Core ML)

**Experiment B — Practical application: MobileCLIP → SigLIP 2 adapter**

Goal: one linear matrix that maps iPhone-tier MobileCLIP-S1 image
embeddings into server-tier SigLIP 2 space, so a **single Qdrant
collection** serves both tiers — the phone indexes images offline, the
server queries the same index with SigLIP text embeddings.


## What this stage does
Exports the trained adapter in two forms:
- `adapter_fp16.npz` (~1 MB) — universal fallback, e.g. for MLX
- `Adapter.mlpackage` — a Core ML model (MatMul + L2-normalize, fp16,
  iOS 16+) that runs on the Neural Engine

## iPhone pipeline after success
`image → MobileCLIP encoder (Apple's official Core ML release) →
Adapter.mlpackage → vector in SigLIP space → shared Qdrant collection`

In Xcode: drag the `.mlpackage` into the project; Swift auto-generates a
class with a single `prediction(mobileclip_embedding:)` call.


In [ ]:
# Storage setup — where stage artifacts (.npz, .png) are read/written.
# Each stage reads the previous stage's output from DATA_DIR.
#
# Option 1 (default): current directory. Works if you run ALL stages in
# the SAME runtime/session. In Colab, a new notebook = a new VM, so files
# from a previous notebook are gone.
#
# Option 2 (Colab, persistent): mount Google Drive and point DATA_DIR
# there — artifacts survive across notebooks and sessions:
#
# from google.colab import drive
# drive.mount('/content/drive')
# os.environ["DATA_DIR"] = "/content/drive/MyDrive/convergence_experiment"

import os
os.environ.setdefault("DATA_DIR", ".")
print("DATA_DIR =", os.path.abspath(os.environ["DATA_DIR"]))

In [ ]:
# Install dependencies (once)
# !pip install coremltools torch

In [ ]:
# Configuration and imports
import numpy as np
import os
from pathlib import Path

DATA_DIR = Path(os.environ.get("DATA_DIR", "."))
DATA_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
# Functions
def main():
    ad = np.load(str(DATA_DIR / "adapter.npz"))
    W = ad["W_ridge"]  # [d_mob, d_sig]

    # 1. fp16 npz (universal fallback, e.g. for MLX)
    np.savez_compressed(str(DATA_DIR / "adapter_fp16.npz"), W=W.astype(np.float16))
    print(f"adapter_fp16.npz: {W.astype(np.float16).nbytes / 1e6:.2f} MB "
          f"({W.shape[0]} -> {W.shape[1]})")

    # 2. Core ML: y = l2norm(x @ W)
    try:
        import torch
        import coremltools as ct

        class Adapter(torch.nn.Module):
            def __init__(self, W):
                super().__init__()
                self.lin = torch.nn.Linear(W.shape[0], W.shape[1],
                                           bias=False)
                self.lin.weight.data = torch.tensor(W.T,
                                                    dtype=torch.float32)

            def forward(self, x):
                y = self.lin(x)
                return torch.nn.functional.normalize(y, dim=-1)

        model = Adapter(W).eval()
        dummy = torch.zeros(1, W.shape[0])
        traced = torch.jit.trace(model, dummy)

        mlmodel = ct.convert(
            traced,
            inputs=[ct.TensorType(name="mobileclip_embedding",
                                  shape=(1, W.shape[0]))],
            outputs=[ct.TensorType(name="siglip_space_embedding")],
            compute_precision=ct.precision.FLOAT16,
            minimum_deployment_target=ct.target.iOS16,
        )
        mlmodel.save(str(DATA_DIR / "Adapter.mlpackage"))
        print("Adapter.mlpackage saved — drop it into your Xcode project.")
        print("Swift usage: let out = try adapter.prediction("
              "mobileclip_embedding: emb)")
    except ImportError as e:
        print(f"Skipped Core ML export ({e}) — adapter_fp16.npz is enough; "
              "run this step on a machine with coremltools installed")

In [ ]:
# Run the export (requires adapter.npz from stage B2)
main()